In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split

In [2]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")


print(train_df.head())


X = train_df.drop(labels=["label"], axis=1).values
y = train_df["label"].values

X = X.reshape(-1 , 1 , 28 , 28)
X = X.astype(np.float32)

X_test = test_df.values.reshape(-1 , 1 , 28 , 28)
X_test = X_test.astype(np.float32)

X = X / 255.0
X_test = X_test / 255.0

X_train , X_val , y_train , y_val = train_test_split(X , y , test_size=0.2 , random_state=42)

train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
val_dataset = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)



   label  pixel0  pixel1  pixel2  pixel3  pixel4  pixel5  pixel6  pixel7  \
0      1       0       0       0       0       0       0       0       0   
1      0       0       0       0       0       0       0       0       0   
2      1       0       0       0       0       0       0       0       0   
3      4       0       0       0       0       0       0       0       0   
4      0       0       0       0       0       0       0       0       0   

   pixel8  ...  pixel774  pixel775  pixel776  pixel777  pixel778  pixel779  \
0       0  ...         0         0         0         0         0         0   
1       0  ...         0         0         0         0         0         0   
2       0  ...         0         0         0         0         0         0   
3       0  ...         0         0         0         0         0         0   
4       0  ...         0         0         0         0         0         0   

   pixel780  pixel781  pixel782  pixel783  
0         0         0         

In [3]:
class DigitClassifierCNN(nn.Module):

    def __init__ (self):
        super().__init__()

        self.features = nn.Sequential(
          
          nn.Conv2d(1, 32, kernel_size=3, padding=1),
          nn.BatchNorm2d(32),
          nn.ReLU(),
          nn.Conv2d(32, 64, kernel_size=3, padding=1),
          nn.BatchNorm2d(64),
          nn.ReLU(),
          nn.MaxPool2d(2, 2),
          nn.Dropout(0.25),

          
          nn.Conv2d(64, 128, kernel_size=3, padding=1),
          nn.BatchNorm2d(128),
          nn.ReLU(),
          nn.Conv2d(128, 128, kernel_size=3, padding=1),
          nn.BatchNorm2d(128),
          nn.ReLU(),
          nn.MaxPool2d(2, 2), 
          nn.Dropout(0.25)
        )

        self.Classifier = nn.Sequential(
            nn.Linear(128 * 7 * 7 , 128),
            nn.ReLU(),
            nn.Dropout(p = 0.4),
            nn.Linear(128 , 10)
        )

    def forward(self , x):
        x = self.features(x)
        x = torch.flatten(x , 1)
        x = self.Classifier(x)
        return x

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DigitClassifierCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters() , lr = 0.001)


epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss , correct_train , total_train = 0.0 , 0 , 0

    for rows,labels in train_loader:
        rows , labels = rows.to(device) , labels.to(device)
        optimizer.zero_grad()
        output = model(rows)
        loss = criterion(output , labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_train += labels.size(0)
        running_loss += loss.item() * rows.size(0)
        _, predicted = torch.max(output.data, 1)

        correct_train += (predicted == labels).sum().item()
    model.eval()

    val_loss , correct_val , total_val= 0.0 , 0 , 0

    with torch.no_grad():

        for rows,labels in val_loader:
            rows , labels = rows.to(device) , labels.to(device)
            output = model(rows)
            loss = criterion(output , labels)

            total_val += labels.size(0)
            val_loss += loss.item()* rows.size(0)
            _, predicted = torch.max(output.data, 1)

            correct_val += (predicted == labels).sum().item()

    train_accuracy = 100 * correct_train / total_train if total_train > 0 else 0.0
    epoch_train_loss = running_loss / total_train

    epoch_val_loss = 0

    if total_val > 0:
        epoch_val_loss = val_loss / total_val
        val_accuracy = 100 * correct_val / total_val
    else:
        epoch_val_loss = float('nan')
        val_accuracy = 0.0


    print("\n" + "="*40)

    print(f" Epoch [{epoch+1}/{epochs}]")
    print(f" Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")


    print(f" Train Accuracy: {train_accuracy:.2f}%")
    print(f" Validation Accuracy: {val_accuracy:.2f}%")
    print("="*40)


KeyboardInterrupt: 

In [ ]:
print(len(X_test))

test_tensor = torch.tensor(X_test, dtype=torch.float32)
test_dataset = TensorDataset(test_tensor)

test_loader = DataLoader(test_dataset , batch_size = 64 , shuffle=False)

model.eval()

all_predictions = []
with torch.no_grad():
  for (rows,) in test_loader:
    rows = rows.to(device)
    output = model(rows)
    _, predicted = torch.max(output, 1)

    all_predictions.extend(predicted.cpu().tolist())

print(f"Total predictions collected: {len(all_predictions)}")

submission_df = pd.DataFrame({
    "ImageId": range(1, len(all_predictions) + 1),
    "Label": all_predictions
})

submission_df.to_csv("submission.csv", index=False)

from google.colab import files

files.download("submission.csv")
